In [1]:
import os, subprocess
import json
import uproot
import awkward as ak
import numpy as np
from coffea import processor, util, hist
import pandas as pd
import pickle

In [2]:
lumis = {}
lumis['2016'] = 35.9
lumis['2017'] = 41.5
lumis['2018'] = 59.9

with open('xsec.json') as f:
  xs = json.load(f)

with open('pmap.json') as f:
  pmap = json.load(f)

In [3]:
year = '2017'
nfiles = len(subprocess.getoutput("ls infiles-split/"+year+"*.json").split())
outsum = processor.dict_accumulator()

In [4]:
repickle=True

# Check if pickle exists, and don't re-create it if it does
picklename = 'pickles/'+str(year)+'_cutflow.pkl'
if os.path.isfile(picklename):
    repickle=False

In [5]:
# Load all files - this takes a while
if repickle:
    nfiles = len(subprocess.getoutput("ls infiles-split/"+year+"*.json").split())
    for n in range(1,nfiles+1):

        with open('infiles-split/'+year+'_'+str(n)+'.json') as f:
            infiles = json.load(f)
    
        filename = '/myeosdir/ggf-vbf/outfiles-ddb2/'+year+'_'+str(n)+'.coffea'
        if os.path.isfile(filename):
            out = util.load(filename)
            outsum.add(out)
        else:
            print('Missing file '+str(n),infiles.keys())
            #print("File " + filename + " is missing")
        
    scale_lumi = {k: xs[k] * 1000 *lumis[year] / w for k, w in outsum['sumw'].items()}
    outsum['cutflow'].scale(scale_lumi, 'dataset')
    cutflow = outsum['cutflow'].group('dataset', hist.Cat('process', 'Process'), pmap)

    outfile = open(picklename, 'wb')
    pickle.dump(cutflow, outfile, protocol=-1)
    outfile.close()

Missing file 2416 dict_keys(['WJetsToLNu_HT-800To1200_TuneCP5_13TeV-madgraphMLM-pythia8'])


/opt/conda/lib/python3.8/site-packages/coffea/hist/hist_tools.py:376: RuntimeWarning: Not all requested indices present in <Cat (name=dataset) instance at 0x7fd3a183e040>
  warnings.warn(


In [6]:
# Read the histogram from the pickle file
templates = pickle.load(open(picklename,'rb'))

In [7]:
cutflow = templates.sum('genflavor').integrate('region','signal-ggf')
cutflow.values()

{('ZH',): array([365.04661246, 365.04661246, 365.04661246,  94.44354574,
         94.44354574,  89.94455609,  54.9181011 ,  26.12645544,
         19.65558786,  16.65618063,  16.45225702,   0.44626031,
          0.        ,   0.        ,   0.        ]),
 ('WH',): array([434.9368986 , 434.9368986 , 434.9368986 , 118.84685093,
        118.84685093, 112.22364368,  65.51732133,  35.13153055,
         24.39392355,  21.28680113,  20.74000139,   0.87501435,
          0.        ,   0.        ,   0.        ]),
 ('ttH',): array([1.15999415e+03, 1.15999415e+03, 1.15999415e+03, 4.36112105e+02,
        4.36112105e+02, 3.67299671e+02, 1.28819052e+02, 4.33668210e+01,
        3.32193958e+01, 2.19852485e+01, 2.14713561e+01, 3.76839808e-01,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]),
 ('VBF',): array([752.21154159, 752.21154159, 752.21154159,  89.2079906 ,
         89.2079906 ,  86.55060599,  47.81515367,  39.33505266,
         37.32167334,  35.9610754 ,  20.56773448,   1.69039227,
        

In [8]:
df1 = pd.DataFrame([])

df1['ggF'] = cutflow.values()[('ggF',)]
df1['VBF'] = cutflow.values()[('VBF',)]
df1['WH'] = cutflow.values()[('WH',)]
df1['ZH'] = cutflow.values()[('ZH',)]
df1['ttH'] = cutflow.values()[('ttH',)]

df1 = df1[3:-3].astype('int')
df1.index = ['Jet kinematics','Jet ID','Jet acceptance','Jet $N_2^\text{DDT}$','Opp. hem. b veto','MET $<$ 140 GeV','No leptons','2 AK4 jets','DeepDoubleB']

df1.to_latex(buf=year+'/cutflow-ggf-sig.tex')

In [9]:
df2 = pd.DataFrame([])

df2['QCD'] = cutflow.values()[('QCD',)]
df2['Wjets'] = cutflow.values()[('Wjets',)]
df2['Zjets'] = cutflow.values()[('Zjets',)]
df2['VV'] = cutflow.values()[('VV',)]
df2['ttbar'] = cutflow.values()[('ttbar',)]
df2['singlet'] = cutflow.values()[('singlet',)]

df2 = df2[3:-3].astype('int')
df2.index = ['Jet kinematics','Jet ID','Jet acceptance','Jet $N_2^\text{DDT}$','Opp. hem. b veto','MET $<$ 140 GeV','No leptons','2 AK4 jets','DeepDoubleB']

df2.to_latex(buf=year+'/cutflow-ggf-bkg.tex')

In [10]:
cutflow = templates.sum('genflavor').integrate('region','signal-vbf')
cutflow.values()

{('ZH',): array([3.65046612e+02, 3.65046612e+02, 3.65046612e+02, 9.44435457e+01,
        9.44435457e+01, 8.99445561e+01, 5.49181011e+01, 2.61264554e+01,
        1.96555879e+01, 1.66561806e+01, 2.03923602e-01, 1.24546446e-02,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]),
 ('WH',): array([4.34936899e+02, 4.34936899e+02, 4.34936899e+02, 1.18846851e+02,
        1.18846851e+02, 1.12223644e+02, 6.55173213e+01, 3.51315306e+01,
        2.43939236e+01, 2.12868011e+01, 5.46799742e-01, 7.35267677e-02,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]),
 ('ttH',): array([1.15999415e+03, 1.15999415e+03, 1.15999415e+03, 4.36112105e+02,
        4.36112105e+02, 3.67299671e+02, 1.28819052e+02, 4.33668210e+01,
        3.32193958e+01, 2.19852485e+01, 5.13892444e-01, 1.06717570e-02,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]),
 ('VBF',): array([752.21154159, 752.21154159, 752.21154159,  89.2079906 ,
         89.2079906 ,  86.55060599,  47.81515367,  39.33505266,
         37.

In [11]:
df1 = pd.DataFrame([])

df1['ggF'] = cutflow.values()[('ggF',)]
df1['VBF'] = cutflow.values()[('VBF',)]
df1['WH'] = cutflow.values()[('WH',)]
df1['ZH'] = cutflow.values()[('ZH',)]
df1['ttH'] = cutflow.values()[('ttH',)]

df1 = df1[3:-3].astype('int')
df1.index = ['Jet kinematics','Jet ID','Jet acceptance','Jet $N_2^\text{DDT}$','Opp. hem. b veto','MET $<$ 140 GeV','No leptons','2 AK4 jets','DeepDoubleB']

df1.to_latex(buf=year+'/cutflow-vbf-sig.tex')

In [12]:
df2 = pd.DataFrame([])

df2['QCD'] = cutflow.values()[('QCD',)]
df2['Wjets'] = cutflow.values()[('Wjets',)]
df2['Zjets'] = cutflow.values()[('Zjets',)]
df2['VV'] = cutflow.values()[('VV',)]
df2['ttbar'] = cutflow.values()[('ttbar',)]
df2['singlet'] = cutflow.values()[('singlet',)]

df2 = df2[3:-3].astype('int')
df2.index = ['Jet kinematics','Jet ID','Jet acceptance','Jet $N_2^\text{DDT}$','Opp. hem. b veto','MET $<$ 140 GeV','No leptons','2 AK4 jets','DeepDoubleB']

df2.to_latex(buf=year+'/cutflow-vbf-bkg.tex')